In [26]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency

# Step1: create dataset
data = pd.DataFrame({
    'study_hours':[2,3,4,5,6,7,8,9],
    'attendance' :[60,65,70,75,80,85,90,95],
    'assignment':['no','no','yes','yes','yes','yes','yes','yes'],
    'result':['fail','fail','fail','pass','pass','pass','pass','pass']
})


print("Original dataset:-")
print(data.head())

#Step2: convert categorical variables
encoder_assignment = LabelEncoder()
data['assignment'] = encoder_assignment.fit_transform(data['assignment'])

encoder_result = LabelEncoder()

data['result'] = encoder_result.fit_transform(data['result'])


x = data[['study_hours','attendance','assignment']]
y = data['result']

print("\nDataset after encoding:-")
print(data.head())


#Step 3: entropy function
def entropy(values):
  probabilities = values.value_counts(normalize = True)
  # Handle case where probabilities might contain 0, which log2(0) is -inf
  # Replace 0 probabilities with a very small number or filter them out to avoid errors
  probabilities = probabilities[probabilities > 0]
  return -np.sum(
      probabilities * np.log2(probabilities)
  )

# Step 4: information gain
def information_gain(feature,target):
  total_entropy = entropy(target)
  unique_values = feature.unique()
  weighted_entropy = 0

# calculate entropy for each feature value
  for value in unique_values:
    subset = target[feature == value]
    entropy_value = entropy(subset)
    proportion = len(subset) / len(target)
    weighted_entropy += proportion * entropy_value

# information gain
  ig_value = total_entropy - weighted_entropy
  return ig_value

# Calculate IG scores
ig_scores = []
for feature_name in x.columns:
  ig_score = information_gain(x[feature_name],y)
  ig_scores.append(ig_score)

ig_result = pd.DataFrame({
  'Feature': x.columns,
  'Information Gain': ig_scores
})

# step 5: gain ratio
def gain_ratio(feature,target):
  information_gain_value = information_gain(feature,target)
  split_info = entropy(feature)
  if split_info == 0:
    return 0 # Avoid division by zero
  gain_ratio_value = information_gain_value / split_info
  return gain_ratio_value

# Calculate GR scores
gr_scores = []
for feature_name in x.columns:
  gr_score = gain_ratio(x[feature_name],y)
  gr_scores.append(gr_score)

gr_result = pd.DataFrame({
  'Feature': x.columns,
  'Gain Ratio': gr_scores
})

#Step 6: prepare data for chi-square sigf
sigf_data = data.copy()
# study hours : low/high
sigf_data['study_hours'] = pd.cut(sigf_data['study_hours'],bins = [0,5,10],labels = ['low','high'], right=False)
# attendance : low/high
sigf_data['attendance'] = pd.cut(sigf_data['attendance'],bins = [0,70,100],labels = ['low','high'], right=False)


# step 7 : sigf using chi square
sigf_results_list = []
features_for_chi_square = ['study_hours','attendance','assignment']
for feature_name in features_for_chi_square:
  contingency_table = pd.crosstab(sigf_data[feature_name],sigf_data['result'])
  chi2,p_value,_,_ = chi2_contingency(contingency_table)
  sigf_results_list.append({'Feature': feature_name, 'Chi2': chi2, 'P_Value': p_value})

sigf_result_df = pd.DataFrame(sigf_results_list)

# step 8: combine resluts
# Merge ig_result, gr_result, and sigf_result_df
result = ig_result.merge(gr_result, on='Feature')
result = result.merge(sigf_result_df, on='Feature')

# step 9: rank features
result['IG Rank'] = result['Information Gain'].rank(ascending = False,method = 'min')
result['GR Rank'] = result['Gain Ratio'].rank(ascending = False,method = 'min')
result['Sigf Rank (Chi2)'] = result['Chi2'].rank(ascending = False,method = 'min')

# Step 10: display feature ranking
print("\nFeature Ranking:-")
print(result[
    ['Feature','IG Rank','GR Rank','Sigf Rank (Chi2)']
].sort_values(by = ['IG Rank','GR Rank','Sigf Rank (Chi2)']))


# step 11: thresholding
ig_threshold = 0.05
gr_threshold = 0.05
p_threshold = 0
sigf_threshold = 0.0
selected_features = result[
    (result['Information Gain'] >= ig_threshold) &
    (result['Gain Ratio'] >= gr_threshold) &
    (result['Chi2'] >= p_threshold)]

# step 12: display selected features
print("\nSelected Features:-")
if len(selected_features) > 0:
  for feature in selected_features['Feature']:
    print(feature)
else:
  print("No features meet the threshold criteria.")



Original dataset:-
   study_hours  attendance assignment result
0            2          60         no   fail
1            3          65         no   fail
2            4          70        yes   fail
3            5          75        yes   pass
4            6          80        yes   pass

Dataset after encoding:-
   study_hours  attendance  assignment  result
0            2          60           0       0
1            3          65           0       0
2            4          70           1       0
3            5          75           1       1
4            6          80           1       1

Feature Ranking:-
       Feature  IG Rank  GR Rank  Sigf Rank (Chi2)
0  study_hours      1.0      2.0               1.0
1   attendance      1.0      2.0               2.0
2   assignment      3.0      1.0               2.0

Selected Features:-
study_hours
attendance
assignment
